# Datenaufbereitung und -bereinigung mit Tidyverse

Sprachübergreifende Daten-Pipeline: Python lädt Daten herunter → R verarbeitet sie mit dplyr → Python visualisiert die Ergebnisse.

Demonstriert **SharedVFS** — das gemeinsam genutzte Dateisystem, mit dem Python und R Dateien austauschen können.

## 1. Python: Datensatz herunterladen

In [ ]:
import micropip
await micropip.install('pandas')
import pandas as pd, pyodide.http, os

url = "https://raw.githubusercontent.com/resbaz/r-novice-gapminder-files/master/data/gapminder-FiveYearData.csv"
resp = await pyodide.http.pyfetch(url)
text = await resp.string()

os.makedirs("/shared/data", exist_ok=True)
with open("/shared/data/gapminder.csv", "w") as f:
    f.write(text)

df = pd.read_csv("/shared/data/gapminder.csv")
print(f"Heruntergeladen: {df.shape[0]} Zeilen, {df.shape[1]} Spalten")
df.head()

## 2. R: dplyr + tidyr installieren und geteilte Daten einlesen

In [ ]:
install.packages(c("dplyr", "tidyr"))
library(dplyr)

gap <- read.csv("/shared/data/gapminder.csv")
cat("Aus SharedVFS eingelesen:", nrow(gap), "Zeilen\n")
glimpse(gap)

## 3. dplyr: Nach Kontinent zusammenfassen (2007)

In [ ]:
gap %>%
  filter(year == 2007) %>%
  group_by(continent) %>%
  summarize(
    countries = n(),
    mean_life = round(mean(lifeExp), 1),
    median_gdp = round(median(gdpPercap), 0),
    total_pop = sum(as.numeric(pop))
  ) %>%
  arrange(desc(mean_life))

## 4. dplyr: Größter Zuwachs an Lebenserwartung

In [ ]:
gains <- gap %>%
  filter(year %in% c(1952, 2007)) %>%
  select(country, continent, year, lifeExp) %>%
  tidyr::pivot_wider(names_from = year, values_from = lifeExp,
                     names_prefix = "y") %>%
  mutate(gain = y2007 - y1952) %>%
  arrange(desc(gain)) %>%
  head(10)
gains

## 5. dplyr: Bevölkerungswachstum nach Kontinent

In [ ]:
pop_growth <- gap %>%
  filter(year %in% c(1952, 2007)) %>%
  group_by(continent, year) %>%
  summarize(total_pop = sum(as.numeric(pop)), .groups = "drop") %>%
  tidyr::pivot_wider(names_from = year, values_from = total_pop,
                     names_prefix = "pop_") %>%
  mutate(growth_pct = round((pop_2007 / pop_1952 - 1) * 100, 1)) %>%
  arrange(desc(growth_pct))
pop_growth

## 6. R: Ergebnisse in SharedVFS schreiben

In [ ]:
# Kontinent-Zusammenfassung für die Visualisierung in Python schreiben
summary_2007 <- gap %>%
  filter(year == 2007) %>%
  group_by(continent) %>%
  summarize(
    mean_life = round(mean(lifeExp), 1),
    mean_gdp = round(mean(gdpPercap), 0),
    .groups = "drop"
  )
write.csv(summary_2007, "/shared/data/r_summary.csv", row.names = FALSE)
cat("/shared/data/r_summary.csv geschrieben\n")
summary_2007

## 7. Python: Ergebnisse von R visualisieren

In [ ]:
import micropip
await micropip.install('plotly')
import plotly.express as px
import json, js
from plotly.utils import PlotlyJSONEncoder

def plain_plotly(value):
    if hasattr(value, 'tolist'):
        return value.tolist()
    if isinstance(value, dict):
        return {key: plain_plotly(item) for key, item in value.items()}
    if isinstance(value, (list, tuple)):
        return [plain_plotly(item) for item in value]
    return value

def show_plotly(fig):
    payload = plain_plotly(fig.to_plotly_json())
    js.renderPlot(json.dumps({"traces": payload["data"], "layout": payload["layout"]}, cls=PlotlyJSONEncoder))

r_summary = pd.read_csv("/shared/data/r_summary.csv")
print("Aus SharedVFS eingelesen (von R geschrieben):")
print(r_summary.to_string(index=False))

fig = px.bar(r_summary, x="continent", y="mean_life",
             title="Mittlere Lebenserwartung nach Kontinent (2007) — von R → Python",
             labels={"mean_life": "Lebenserwartung (Jahre)", "continent": "Kontinent"},
             color="continent")
fig.update_layout(template='plotly_dark', showlegend=False)
show_plotly(fig)

In [ ]:
fig = px.scatter(r_summary, x="mean_gdp", y="mean_life",
                 text="continent", size=[40]*len(r_summary),
                 title="BIP vs. Lebenserwartung nach Kontinent (R-Zusammenfassung → Python-Diagramm)",
                 labels={"mean_gdp": "Mittleres BIP pro Kopf", "mean_life": "Mittlere Lebenserwartung"})
fig.update_traces(textposition="top center")
fig.update_layout(template='plotly_dark')
fig.update_yaxes(range=[r_summary['mean_life'].min() - 2, r_summary['mean_life'].max() + 6])
show_plotly(fig)

## Wichtigste Erkenntnisse

- **Python** hat CSV-Daten nach `/shared/data/` heruntergeladen
- **R** hat sie über SharedVFS eingelesen und mit dplyr-Pipelines verarbeitet
- **R** hat die Zusammenfassung zurück nach `/shared/data/r_summary.csv` geschrieben
- **Python** hat die Ausgabe von R gelesen und interaktive Plotly-Diagramme erstellt

Der gesamte Dateiaustausch erfolgt über SharedVFS — kein manuelles Importieren oder Exportieren erforderlich.